# 30 — Recommended private-encoder MoE, three seeds
Runs one locked architecture on seeds 0, 1, and 2. Validation macro-F1 selects each seed's Stage-C epoch; test data is evaluated only after the configuration is fixed. Every seed saves A/B/C latent metrics, embeddings, PCA/UMAP plots, and per-class focus figures. This workflow is independent from notebooks 18–24 and the targeted-depth notebooks.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'
STUDY_CONFIG = 'config/recommended_private_3seed.yaml'
PREFIX = 'nfv3_4way_recommended_private_bsupcon_v1'
EXECUTE = False  # First run the dry-run; then change this to True.
MAX_NEW_SEEDS = 3  # Use 1 for one resumable seed per Colab session.
RUN_TESTS = True
# ==================================================================

## Secure checkout and environment setup

In [ ]:
import base64, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata
drive.mount('/content/drive')
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant access.')
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git_env = os.environ | {
    'GIT_CONFIG_COUNT': '1',
    'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
    'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
}
repo = Path('/content') / GITHUB_REPO
url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', url, str(repo)], env=git_env, check=True)
git_env.clear(); token = auth = None
os.chdir(repo)
os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)

## Preflight, dry-run, and resumable execution

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before executing the study.')
if RUN_TESTS:
    subprocess.run([
        sys.executable, '-m', 'pytest', '-q',
        'tests/test_recommended_private_run.py', 'tests/test_latent_space.py',
        'tests/test_ablation_study.py',
    ], check=True)
command = [
    sys.executable, '-u', '-m', 'training.recommended_private_run',
    '--study-config', STUDY_CONFIG, '--prefix', PREFIX,
]
print('=== Complete dry-run graph ===', flush=True)
subprocess.run(command, check=True, env=os.environ.copy())
if EXECUTE:
    launch = [*command, '--execute', '--max-new-seeds', str(MAX_NEW_SEEDS)]
    print('=== Executing fixed three-seed workflow ===', flush=True)
    subprocess.run(launch, check=True, env=os.environ.copy())
else:
    print('Dry-run only. Review the graph, set EXECUTE=True, and rerun this cell.')

## Three-seed task and guardrail summaries

In [ ]:
import pandas as pd
from IPython.display import Image, display
summary_dir = Path(DRIVE_OUTPUT_DIR) / 'results' / f'{PREFIX}_summary'
manifest_path = summary_dir / 'Final_Study_Manifest.json'
if not manifest_path.is_file():
    print('The three-seed run is not complete yet:', summary_dir)
else:
    manifest = json.loads(manifest_path.read_text())
    print(json.dumps(manifest, indent=2))
    for filename in [
        'Three_Seed_Overall_Summary.csv',
        'Three_Seed_Guardrails.csv',
        'Three_Seed_Per_Dataset_Summary.csv',
        'Three_Seed_Per_Class_Summary.csv',
        'Three_Seed_Resource_Accounting.csv',
    ]:
        print('===', filename, '===')
        display(pd.read_csv(summary_dir / filename))

## Latent-space metrics and saved figures

In [ ]:
if manifest_path.is_file():
    for filename in [
        'Latent_Snapshot_Summary_3Seed.csv',
        'Latent_Probe_Summary_3Seed.csv',
        'Latent_Per_Class_Summary_3Seed.csv',
    ]:
        print('===', filename, '===')
        display(pd.read_csv(summary_dir / filename))
    summary_figures = sorted((summary_dir / 'latent_figures').glob('*.png'))
    for path in summary_figures:
        print(path.name)
        display(Image(filename=str(path)))
    seed0 = Path(manifest['result_dirs']['0']) / 'latent' / 'cumulative' / 'latent_figures'
    representative = [
        seed0 / 'A__private_initialization__umap.png',
        seed0 / 'B__expert_NF-UNSW-NB15-v3__umap.png',
        seed0 / 'C__expert_NF-UNSW-NB15-v3__umap.png',
    ]
    print('=== Representative seed-0 UMAP gallery ===')
    for path in representative:
        if path.is_file():
            display(Image(filename=str(path)))
    print('All per-seed PCA, UMAP, class-focus, metric, manifest, and embedding files are under:')
    for seed, path in manifest['result_dirs'].items():
        print(f'  seed {seed}:', Path(path) / 'latent')